In [2]:
import json
import struct

In [3]:
page_size = 16 * 1024

#The origin of a record points to the beginning of *data* in the record, 
#   not to the beginning of the record itself.
#The infimum record has a fixed size, with header size = 5 bytes.
#   The record header contains metadata about the record, 
#       such as the type of the record.
#   The data of the infimum record is 8 bytes, as shown in class.
#See the following link for more details:
#   https://blog.jcole.us/2013/01/07/the-physical-structure-of-innodb-index-pages/
infimum_record_origin = 94 + 5

In [4]:
def get_index(indexes: list, index_id: int, page_no) -> dict:
    """Returns the index with the given index_id from the list of indexes."""
    for index in indexes:
        if index['index_id'] == index_id and index['root'] == page_no:
            return index

In [5]:
def decode_innodb_float(b: bytes):
    """
    Decode an InnoDB-encoded float (unsigned float is not supported)
    """
    if len(b) != 4:
        raise ValueError("Unsupported float length")

    # innodb stores float in little-endian format
    return struct.unpack('<f', b)[0]

In [ ]:
def decode_innodb_double(b: bytes):
    """
    Decode an InnoDB-encoded double (unsigned double is not supported)
    """
    ######################################
    ## fill in your code here (10 points)
    ######################################

In [7]:
def decode_innodb_signed_int(b: bytes) -> int:
    """
    Decode an InnoDB-encoded signed integer of any size:
    1, 2, 3, 4, or 8 bytes.
    """
    length = len(b)
    if length not in (1, 2, 3, 4, 8):
        raise ValueError("Unsupported integer length")

    bits = length * 8

    # Step 1: read as unsigned big-endian
    stored = int.from_bytes(b, byteorder="big", signed=False)

    # Step 2: undo sign-bit flip
    sign_mask = 1 << (bits - 1)
    original = stored ^ sign_mask

    # Step 3: convert to signed range
    if original >= (1 << (bits - 1)):
        original -= (1 << bits)

    return original


In [ ]:
def extract_one_record(page, offset, fields):
    """Extracts one record from the page starting at the given offset,
    and returns a dictionary of field values."""

    field_values = {}
    
    ######################################
    ## fill in your code here (60 points)
    ######################################
    
    return field_values


In [9]:
def extract_records_for_index(page: bytes, index: dict) -> list:   
    """Extracts all records for the given index from the page, 
        and returns a list of dictionaries of field values."""
    #print('* Index', index['name'])
    
    offset = infimum_record_origin

    # Read 2 bytes for relative offset of first record
    first_rel = int.from_bytes(page[offset-2:offset])  
    #print(f"First relative offset: {first_rel}")
    offset = offset + first_rel  # Move to the first record
    #print(f"First record offset: {offset}")

    record_no = 0
    records = []

    while True:
            rel = int.from_bytes(page[offset-2:offset])  
            
            if rel == 0: # we reach supremum record,
                # which is the last record in the page
                #print("No more records to read.")
                break
            
            # read a record at the current offset
            record = extract_one_record(page, offset, index['records'][record_no])
            #print(record)
            records.append(record)

            record_no += 1
            
            # Move to the next record, with 16-bit wraparound
            offset = (offset + rel) & 0xFFFF  
            #print(f"Next record offset: {offset}")
            
    return records

In [10]:
def extract_indexes_from_ibd(ibd_file: str, decoding_file: str):
    """
    Extract index information from an InnoDB .ibd file using the provided 
    decoding file.The decoding file contains an JSON array of objects where 
    each object shows how to extract records/entries for a specific index. 
    Each object has the following structure:
        {
            "index_id": 565,
            "name": "PRIMARY",
            "records": [
                [ # this is the first record/entry of the index
                    {
                    "name": "id",
                    "type": "int",
                    "size": 4,
                    "is_null": false
                    }, # this is the first field of the first record/entry
                    ...
                ], 
                ...
            ]
        }

    """
    
    with open(decoding_file, 'r') as f:
        indexes = json.load(f)

    index_data = {}

    with open(ibd_file, 'rb') as f:
        while True:
            page = f.read(page_size)  # Read 16KB pages
            if not page:
                break

            #Page type is at offset 24-25
            page_type = int.from_bytes(page[24:26]) 
            #Page no is at offset 4-7
            page_no = int.from_bytes(page[4:8])

            if page_type == 0x45BF:  # This is an index page
                # Extract the index_id from the page header (offset 26-29)
                index_id = int.from_bytes(page[66:74])
                #print('\tindex_id', index_id)
                
                index = get_index(indexes, index_id, page_no)
                if index is None:   # skip this page
                    continue

                records = extract_records_for_index(page, index)
                #print(json.dumps(records, indent=2))

                index_data[index['name']] = records

    return index_data

In [11]:
result = extract_indexes_from_ibd('user.ibd', 'user_decoding.json')
print(json.dumps(result, indent=2))


{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b019",
      "DB_ROLL_PTR": "0x81000000e20110",
      "name": "john"
    },
    {
      "id": 101,
      "DB_TRX_ID": "0x00000000b01a",
      "DB_ROLL_PTR": "0x82000000e20110",
      "name": "david"
    }
  ],
  "name_idx": [
    {
      "name": "david",
      "id": 101
    },
    {
      "name": "john",
      "id": 100
    }
  ]
}


In [12]:
result = extract_indexes_from_ibd('test.ibd', 'test_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "DB_ROW_ID": "0x0000000dc9fb",
      "DB_TRX_ID": "0x00000000b04e",
      "DB_ROLL_PTR": "0x81000000ff0110",
      "a": 100,
      "b": "bill",
      "c": 1
    },
    {
      "DB_ROW_ID": "0x0000000dc9fc",
      "DB_TRX_ID": "0x00000000b04f",
      "DB_ROLL_PTR": "0x82000001210110",
      "a": null,
      "b": "david",
      "c": 2
    },
    {
      "DB_ROW_ID": "0x0000000dc9fd",
      "DB_TRX_ID": "0x00000000b054",
      "DB_ROLL_PTR": "0x810000012e0110",
      "a": null,
      "b": null,
      "c": 4
    },
    {
      "DB_ROW_ID": "0x0000000dc9fe",
      "DB_TRX_ID": "0x00000000b055",
      "DB_ROLL_PTR": "0x82000001230110",
      "a": 101,
      "b": null,
      "c": null
    }
  ]
}


In [13]:
result = extract_indexes_from_ibd('student.ibd', 'student_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b037",
      "DB_ROLL_PTR": "0x81000000f40110",
      "name": "john",
      "gender": "male"
    },
    {
      "id": 101,
      "DB_TRX_ID": "0x00000000b038",
      "DB_ROLL_PTR": "0x82000001530110",
      "name": "mary",
      "gender": "female"
    },
    {
      "id": 102,
      "DB_TRX_ID": "0x00000000b03d",
      "DB_ROLL_PTR": "0x81000001570110",
      "name": "david",
      "gender": null
    }
  ]
}


In [14]:
result = extract_indexes_from_ibd('test_types.ibd', 'test_types_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b067",
      "DB_ROLL_PTR": "0x810000011c0110",
      "id1": 101,
      "age": 25,
      "age1": 26,
      "age2": 27,
      "age3": 28,
      "score": 1000,
      "name": "john smith",
      "gpa": 4.5,
      "salary": 1000.8,
      "height": "0x80af1c",
      "addr": "100 maple st",
      "dob": "0x8fd422",
      "resume": "my cv is text type"
    }
  ],
  "name": [
    {
      "name": "john smith",
      "id": 100
    }
  ]
}


In [15]:
result = extract_indexes_from_ibd('tbl1.ibd', 'tbl1_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "a": "bill",
      "b": "david     ",
      "DB_TRX_ID": "0x00000000b091",
      "DB_ROLL_PTR": "0x81000001210110",
      "c": -2
    },
    {
      "a": "david",
      "b": "john      ",
      "DB_TRX_ID": "0x00000000b08c",
      "DB_ROLL_PTR": "0x820000010f0110",
      "c": null
    },
    {
      "a": "john",
      "b": "david     ",
      "DB_TRX_ID": "0x00000000b08b",
      "DB_ROLL_PTR": "0x81000001120110",
      "c": 2
    }
  ]
}


In [16]:
result = extract_indexes_from_ibd('employee.ibd', 'employee_decoding.json')
print(json.dumps(result, indent=2))

{
  "PRIMARY": [
    {
      "id": 100,
      "DB_TRX_ID": "0x00000000b078",
      "DB_ROLL_PTR": "0x810000010b0110",
      "name": "john      ",
      "addr": "100 maple street"
    },
    {
      "id": 101,
      "DB_TRX_ID": "0x00000000b079",
      "DB_ROLL_PTR": "0x82000001170110",
      "name": "bill chu  ",
      "addr": "200 vermont av, LA"
    },
    {
      "id": 102,
      "DB_TRX_ID": "0x00000000b07a",
      "DB_ROLL_PTR": "0x81000002170110",
      "name": null,
      "addr": "123 main blvd."
    }
  ]
}
